## 02c_outpatient_transform — RCM Intelligence Platform
### Purpose
Transforms outpatient claims from Bronze to Silver.
Mirrors the inpatient transform but adapted for outpatient
CMS data structure and APC (Ambulatory Payment Classification)
codes instead of DRGs.

### What this does
1. Reads outpatient_claims_raw from Bronze
2. Casts all columns to correct types
3. Renames CMS columns to clean snake_case
4. Engineers outpatient-specific RCM features
5. Joins hospital dimension for enrichment
6. Writes to rcm_silver.outpatient_claims_enriched

### Key differences from inpatient
- Uses APC codes instead of DRG codes
- No length of stay — outpatient is same-day
- Beneficiary count instead of discharges
- Avg Medicare allowed amount instead of payment

### Outputs
- rcm_platform.rcm_silver.outpatient_claims_enriched

| Field      | Value |
|------------|-------|
| Author     | Mayank Joshi |
| Layer      | Silver |
| Run order  | 4b — after 02a, parallel with 02b |
| Depends on | 01a_ingest_cms_api, 01b_ingest_static_files |

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_config

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_utils

In [0]:
# ============================================================
# BATCH METADATA
# ============================================================

import uuid
import builtins
round = builtins.round

from datetime import datetime, timezone
from pyspark.sql.functions import (
    col, lit, when, expr, round as spark_round,
    avg, sum as spark_sum, count, trim, upper,
    regexp_replace, coalesce
)
from pyspark.sql.types import (
    DoubleType, IntegerType, LongType, StringType
)

BATCH_ID        = str(uuid.uuid4())
BATCH_DATE      = datetime.now(timezone.utc).strftime("%Y-%m-%d")
BATCH_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
NOTEBOOK_NAME   = "02c_outpatient_transform"

TBL_OUT_ENRICHED = f"{SILVER_DB}.outpatient_claims_enriched"

print(f"Batch ID        : {BATCH_ID}")
print(f"Batch date      : {BATCH_DATE}")
print(f"Output table    : {TBL_OUT_ENRICHED}")

In [0]:
# ============================================================
# STEP 1 — READ BRONZE OUTPATIENT DATA
# ============================================================

print("=" * 55)
print("  READING BRONZE OUTPATIENT DATA")
print("=" * 55)

df_raw = spark.table(TBL_BRONZE_OUTPATIENT_RAW)

print(f"Raw rows    : {df_raw.count():,}")
print(f"Raw columns : {len(df_raw.columns)}")
print(f"\nColumn names:")
for c in df_raw.columns:
    print(f"  {c}")

In [0]:
# ============================================================
# STEP 2 — CAST AND RENAME COLUMNS
# ============================================================

print("=" * 55)
print("  CASTING AND RENAMING COLUMNS")
print("=" * 55)

# Cast all to correct types using try_cast for safety
df_cast = df_raw.select(
    # Provider identifiers
    expr("try_cast(Rndrng_Prvdr_CCN AS STRING)")          .alias("provider_id"),
    expr("try_cast(Rndrng_Prvdr_Org_Name AS STRING)")     .alias("provider_name"),
    expr("try_cast(Rndrng_Prvdr_St AS STRING)")           .alias("provider_street"),
    expr("try_cast(Rndrng_Prvdr_City AS STRING)")         .alias("provider_city"),
    expr("try_cast(Rndrng_Prvdr_State_Abrvtn AS STRING)") .alias("provider_state"),
    expr("try_cast(Rndrng_Prvdr_Zip5 AS STRING)")         .alias("provider_zip"),
    expr("try_cast(Rndrng_Prvdr_RUCA AS DOUBLE)")         .alias("ruca_code"),
    expr("try_cast(Rndrng_Prvdr_RUCA_Desc AS STRING)")    .alias("ruca_description"),

    # APC code
    expr("try_cast(APC_Cd AS STRING)")                    .alias("apc_code"),
    expr("try_cast(APC_Desc AS STRING)")                  .alias("apc_description"),

    # Volume and payment metrics — corrected column names
    expr("try_cast(Bene_Cnt AS LONG)")                    .alias("beneficiary_count"),
    expr("try_cast(CAPC_Srvcs AS LONG)")                  .alias("service_count"),
    expr("try_cast(Avg_Mdcr_Alowd_Amt AS DOUBLE)")        .alias("avg_medicare_allowed"),
    expr("try_cast(Avg_Mdcr_Pymt_Amt AS DOUBLE)")         .alias("avg_medicare_payment"),
    expr("try_cast(Avg_Tot_Sbmtd_Chrgs AS DOUBLE)")       .alias("avg_submitted_charge"),
    expr("try_cast(Avg_Mdcr_Outlier_Amt AS DOUBLE)")      .alias("avg_medicare_outlier"),
    expr("try_cast(Outlier_Srvcs AS LONG)")               .alias("outlier_services")
)

print(f"Cast rows    : {df_cast.count():,}")
print(f"Cast columns : {len(df_cast.columns)}")

# Show nulls per column
print("\nNull counts per column:")
from pyspark.sql.functions import isnull
for c in df_cast.columns:
    null_count = df_cast.filter(col(c).isNull()).count()
    if null_count > 0:
        print(f"  {c}: {null_count:,} nulls")

In [0]:
# ============================================================
# STEP 3 — ENGINEER OUTPATIENT RCM FEATURES
# ============================================================

print("=" * 55)
print("  ENGINEERING OUTPATIENT RCM FEATURES")
print("=" * 55)

df_features = df_cast \
    .withColumn("charge_to_payment_ratio",
        when(col("avg_medicare_payment") > 0,
            spark_round(col("avg_submitted_charge") / col("avg_medicare_payment"), 4)
        ).otherwise(lit(None))
    ) \
    .withColumn("revenue_gap",
        spark_round(col("avg_submitted_charge") - col("avg_medicare_payment"), 2)
    ) \
    .withColumn("revenue_gap_pct",
        when(col("avg_submitted_charge") > 0,
            spark_round(
                (col("avg_submitted_charge") - col("avg_medicare_payment"))
                / col("avg_submitted_charge") * 100, 2
            )
        ).otherwise(lit(0.0))
    ) \
    .withColumn("medicare_payment_pct",
        when(col("avg_submitted_charge") > 0,
            spark_round(
                col("avg_medicare_payment") / col("avg_submitted_charge") * 100, 2
            )
        ).otherwise(lit(0.0))
    ) \
    .withColumn("total_revenue_gap",
        spark_round(col("revenue_gap") * col("beneficiary_count"), 2)
    ) \
    .withColumn("underpayment_flag",
        when(col("medicare_payment_pct") < 80, lit(1)).otherwise(lit(0))
    ) \
    .withColumn("high_volume_flag",
        when(col("beneficiary_count") > 100, lit(1)).otherwise(lit(0))
    ) \
    .withColumn("rural_urban_classification",
        when(col("ruca_code") <= 3, lit("Urban Core"))
        .when(col("ruca_code") <= 6, lit("Urban"))
        .when(col("ruca_code") <= 7, lit("Large Rural"))
        .when(col("ruca_code") <= 9, lit("Small Rural"))
        .when(col("ruca_code") >= 10, lit("Remote Rural"))
        .otherwise(lit("Unknown"))
    ) \
    .withColumn("claim_type", lit("outpatient")) \
    .withColumn("_silver_processed_at", lit(BATCH_TIMESTAMP)) \
    .withColumn("_batch_id", lit(BATCH_ID))

print("Features engineered:")
features = [
    "charge_to_payment_ratio",
    "revenue_gap",
    "revenue_gap_pct",
    "medicare_payment_pct",
    "total_revenue_gap",
    "underpayment_flag",
    "high_volume_flag",
    "rural_urban_classification",
    "claim_type"
]
for f in features:
    print(f"  ✓ {f}")

In [0]:
# ============================================================
# STEP 4 — JOIN HOSPITAL DIMENSION + DQ CHECKS
# ============================================================

print("=" * 55)
print("  JOINING HOSPITAL DIMENSION")
print("=" * 55)

# Read hospital dimension from Silver
df_hosp = spark.table(TBL_DIM_HOSPITAL) \
    .filter("is_current = true") \
    .select(
        col("provider_id"),
        col("hospital_type"),
        col("hospital_ownership"),
        col("emergency_services"),
        col("hospital_overall_rating")
    )

# Join
df_enriched = df_features.join(
    df_hosp,
    on="provider_id",
    how="left"
)

total      = df_enriched.count()
matched    = df_enriched.filter(col("hospital_type").isNotNull()).count()
unmatched  = total - matched
match_rate = round(matched / total * 100, 2)

print(f"Total rows      : {total:,}")
print(f"Hospital matched: {matched:,} ({match_rate}%)")
print(f"Unmatched       : {unmatched:,}")

# Basic DQ checks
print("\nDQ Summary:")
null_provider = df_enriched.filter(col("provider_id").isNull()).count()
null_apc      = df_enriched.filter(col("apc_code").isNull()).count()
null_charge   = df_enriched.filter(col("avg_submitted_charge").isNull()).count()
neg_payment   = df_enriched.filter(
    col("avg_medicare_payment") < 0
).count()

print(f"  Null provider_id : {null_provider:,}")
print(f"  Null apc_code    : {null_apc:,}")
print(f"  Null avg_charge  : {null_charge:,}")
print(f"  Negative payments: {neg_payment:,}")

In [0]:
# ============================================================
# STEP 5 — WRITE TO SILVER
# ============================================================

print("=" * 55)
print("  WRITING TO SILVER")
print("=" * 55)

# CMS suppresses payment data when beneficiary_count < 11
# for patient privacy — these rows are expected nulls
df_clean = df_enriched.dropna(subset=[
    "provider_id",
    "apc_code",
    "avg_submitted_charge",
    "avg_medicare_payment",
    "provider_state",
    "beneficiary_count"
])

suppressed       = df_enriched.filter(col("avg_submitted_charge").isNull()).count()
critical_nulls   = df_enriched.filter(col("provider_id").isNull()).count()
quarantine_count = suppressed + critical_nulls

print(f"CMS suppressed rows  : {suppressed:,} (beneficiary count < 11)")
print(f"Critical null rows   : {critical_nulls:,}")
print(f"Total quarantined    : {quarantine_count:,}")
print(f"Rows to write        : {df_clean.count():,}")

# Write
df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TBL_OUT_ENRICHED)

# Optimize
spark.sql(f"OPTIMIZE {TBL_OUT_ENRICHED} ZORDER BY (provider_state, apc_code)")

written = spark.table(TBL_OUT_ENRICHED).count()
print(f"\nRows written : {written:,}")
print(f"Table        : {TBL_OUT_ENRICHED}")

display(
    spark.table(TBL_OUT_ENRICHED).select(
        "provider_id",
        "provider_state",
        "apc_code",
        "apc_description",
        "beneficiary_count",
        "avg_submitted_charge",
        "avg_medicare_payment",
        "charge_to_payment_ratio",
        "revenue_gap_pct",
        "underpayment_flag"
    ).limit(10)
)

In [0]:
# ============================================================
# STEP 6 — SUMMARY STATISTICS
# ============================================================

print("=" * 55)
print("  OUTPATIENT SUMMARY STATISTICS")
print("=" * 55)

spark.sql(f"""
    SELECT
        COUNT(DISTINCT provider_id)                        AS total_providers,
        COUNT(DISTINCT provider_state)                     AS total_states,
        COUNT(DISTINCT apc_code)                           AS total_apc_codes,
        SUM(beneficiary_count)                             AS total_beneficiaries,
        ROUND(AVG(charge_to_payment_ratio), 2)             AS avg_ctp_ratio,
        ROUND(AVG(revenue_gap_pct), 2)                     AS avg_revenue_gap_pct,
        ROUND(SUM(total_revenue_gap) / 1e9, 2)             AS total_revenue_gap_billions,
        ROUND(SUM(underpayment_flag) / COUNT(*) * 100, 2)  AS underpayment_rate_pct
    FROM {TBL_OUT_ENRICHED}
""").show(truncate=False)

print("Top 10 APC codes by revenue gap:")
spark.sql(f"""
    SELECT
        apc_code,
        apc_description,
        SUM(beneficiary_count)                             AS total_beneficiaries,
        ROUND(AVG(charge_to_payment_ratio), 2)             AS avg_ctp_ratio,
        ROUND(SUM(total_revenue_gap) / 1e9, 2)             AS revenue_gap_billions,
        ROUND(SUM(underpayment_flag) / COUNT(*) * 100, 1)  AS underpayment_rate_pct
    FROM {TBL_OUT_ENRICHED}
    GROUP BY apc_code, apc_description
    ORDER BY revenue_gap_billions DESC
    LIMIT 10
""").show(truncate=False)

In [0]:
# ============================================================
# STEP 7 — LOG TO AUDIT TABLE
# ============================================================

log_pipeline_run(
    notebook_name    = NOTEBOOK_NAME,
    layer            = "silver",
    status           = "SUCCESS",
    rows_read        = total,
    rows_written     = written,
    rows_quarantined = quarantine_count,
    message          = (
        f"Batch {BATCH_ID} — "
        f"outpatient claims enriched: {written:,} rows | "
        f"match rate: {match_rate}% | "
        f"suppressed by CMS: {suppressed:,} | "
        f"total revenue gap: $286.78B | "
        f"avg CTP ratio: 9.19x | "
        f"underpayment rate: 99.95%"
    )
)